<a href="https://colab.research.google.com/github/fandri-indranata/data-science-2026/blob/main/Pertemuan12_FandriIndranata_230401010180.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nama: Fandri Indranata
NIM: 230401010180
Kelas: IF401

In [ ]:
import pandas as pd
import numpy as np

# Set seed agar hasil random konsisten
np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur', 'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

# Buat 50 transaksi, tiap transaksi berisi 2-5 produk
transaksi = []
for _ in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(list(np.random.choice(produk, n_item, replace=False)))

# Suntikkan pola bisnis: Roti sering dibeli bersama Selai
for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

print("=== Contoh 5 Transaksi Pertama ===")
for i, t in enumerate(transaksi[:5]):
    print(f"Transaksi {i+1}: {t}")
print(f"\nTotal Transaksi: {len(transaksi)}")

In [ ]:
from mlxtend.preprocessing import TransactionEncoder

# Inisialisasi encoder
te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)

# Ubah ke DataFrame
df_trans = pd.DataFrame(te_ary, columns=te.columns_)

print("=== Hasil One-Hot Encoding (5 Baris Pertama) ===")
print(df_trans.head())

In [ ]:
from mlxtend.frequent_patterns import apriori

print("=== Uji Coba Min Support ===")
for ms in [0.05, 0.1, 0.2]:
    freq_test = apriori(df_trans, min_support=ms, use_colnames=True)
    print(f"min_support = {ms} : {len(freq_test)} itemset ditemukan")

# Gunakan min_support = 0.1 sebagai nilai optimal untuk dataset ini
print("\n=== Top 10 Frequent Itemset (min_support=0.1) ===")
freq_items = apriori(df_trans, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)
print(freq_items.head(10))

In [ ]:
from mlxtend.frequent_patterns import association_rules

# Bentuk aturan dengan min_confidence = 0.5
rules = association_rules(freq_items, metric='confidence', min_threshold=0.5)

# Saring hanya yang lift > 1, lalu urutkan
rules_filtered = rules[rules['lift'] > 1].sort_values('lift', ascending=False)

print("=== Aturan Asosiasi Terkuat (Lift > 1) ===")
# Tampilkan kolom penting saja agar mudah dibaca
cols_tampil = ['antecedents', 'consequents', 'support', 'confidence', 'lift']
print(rules_filtered[cols_tampil].head(10))

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# 1. Buat Katalog Produk dengan Kategori
katalog = pd.DataFrame({
    'produk': produk,
    'kategori': ['Bakery', 'Bakery', 'Dairy', 'Bakery', 'Dairy',
                 'Dairy', 'Minuman', 'Bumbu', 'Minuman', 'Dairy']
})

# 2. One-Hot Encoding pada fitur 'kategori'
fitur_kategori = pd.get_dummies(katalog['kategori'])

# 3. Hitung Cosine Similarity antar produk berdasarkan kategori
sim_matrix = cosine_similarity(fitur_kategori)

# 4. Fungsi untuk mencari produk serupa
def rekomendasi_serupa(nama_produk, top_n=3):
    # Cari indeks produk target
    idx = katalog.index[katalog['produk'] == nama_produk][0]

    # Ambil skor similaritas produk tersebut dengan semua produk lain
    skor = list(enumerate(sim_matrix[idx]))

    # Urutkan dari yang paling mirip (nilai tertinggi), abaikan produk itu sendiri (idx)
    skor = sorted(skor, key=lambda x: x[1], reverse=True)
    skor = [s for s in skor if s[0] != idx][:top_n]

    # Kembalikan nama produk yang paling mirip
    return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()

print("=== Rekomendasi Content-Based untuk 'Roti' ===")
print(rekomendasi_serupa('Roti', top_n=3))

In [ ]:
produk_target = 'Roti'

# 1. Dari Association Rules: cari consequents jika antecedent mengandung 'Roti'
# Kita ubah frozenset menjadi string agar mudah di-filter
rules_terkait = rules_filtered[rules_filtered['antecedents'].apply(lambda x: produk_target in x)]

print(f"=== Perbandingan Rekomendasi untuk Produk: '{produk_target}' ===\n")

print("1. Rekomendasi dari Association Rules (Market Basket):")
if not rules_terkait.empty:
    print(rules_terkait[['consequents', 'lift']].head(3).to_string(index=False))
else:
    print("Tidak ada aturan yang ditemukan untuk produk ini.")

print("\n2. Rekomendasi dari Content-Based Filtering (Kemiripan Kategori):")
print(rekomendasi_serupa(produk_target, top_n=3))